In [1]:
from pathlib import Path
import pandas as pd

# ==========================================================
# Load and Prepare Data
# ==========================================================
# Load cleaned datasets from processed folder
processed_dir = Path("data/processed")
sales_clean = pd.read_csv(processed_dir / "sales_clean.csv")
future_clean = pd.read_csv(processed_dir / "future_clean.csv")

# Convert date columns to datetime
sales_clean["date"] = pd.to_datetime(sales_clean["date"])
future_clean["date"] = pd.to_datetime(future_clean["date"])



C:\Users\setue\AppData\Local\Temp\ipykernel_6692\3603867740.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_clean = pd.read_csv(processed_dir / "sales_clean.csv")


In [2]:
# ==========================================================
# Feature Engineering
# ==========================================================
# Machine learning models cannot directly understand dates.
# We transform the date into calendar numerical features
# to help the model learn weekly and seasonal sales patterns.
# ==========================================================

# Copy datasets to avoid modifying the cleaned data
sales_fe = sales_clean.copy()
future_fe = future_clean.copy()

In [3]:
# ==========================================================
# Clean state_holiday categories
# ==========================================================

sales_fe["state_holiday"] = (
    sales_fe["state_holiday"]
    .astype(str)
    .replace({
        "0.0": "0",
        "0.1": "0"
    })
)

future_fe["state_holiday"] = (
    future_fe["state_holiday"]
    .astype(str)
    .replace({
        "0.0": "0",
        "0.1": "0"
    })
)

In [4]:
# ==========================================================
# Calendar Features
# ==========================================================
# Machine learning models cannot directly use datetime
# variables. Therefore, we extract calendar-based features
# to capture weekly, monthly and yearly seasonal patterns.
#
# Features:
# year         : Calendar year
# month        : Month of the year (1-12)
# quarter      : Quarter of the year (1-4)
# week_of_year : ISO week number (1-52)
# day_of_week  : Day of the week (Monday=0, Sunday=6)
# day_of_month : Day of the month (1-31)
# is_weekend   : Weekend indicator (0=Weekday, 1=Weekend)
# ==========================================================

calendar_features = [
    "year",
    "month",
    "quarter",
    "week_of_year",
    "day_of_week",
    "day_of_month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "is_quarter_end"
]

# ==========================================================
# Generate Calendar Features
# ==========================================================

sales_fe["year"] = sales_fe["date"].dt.year

sales_fe["month"] = sales_fe["date"].dt.month

sales_fe["quarter"] = sales_fe["date"].dt.quarter

sales_fe["week_of_year"] = (
    sales_fe["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

sales_fe["day_of_week"] = sales_fe["date"].dt.dayofweek

sales_fe["day_of_month"] = sales_fe["date"].dt.day

sales_fe["is_weekend"] = (
        sales_fe["day_of_week"] >= 5
).astype(int)

sales_fe["is_month_start"] = (
    sales_fe["date"]
    .dt.is_month_start
    .astype(int)
)

future_fe["is_month_start"] = (
    future_fe["date"]
    .dt.is_month_start
    .astype(int)
)

sales_fe["is_month_end"] = (
    sales_fe["date"]
    .dt.is_month_end
    .astype(int)
)

future_fe["is_month_end"] = (
    future_fe["date"]
    .dt.is_month_end
    .astype(int)
)

sales_fe["is_quarter_end"] = (
    sales_fe["date"]
    .dt.is_quarter_end
    .astype(int)
)

future_fe["is_quarter_end"] = (
    future_fe["date"]
    .dt.is_quarter_end
    .astype(int)
)


future_fe["year"] = future_fe["date"].dt.year

future_fe["month"] = future_fe["date"].dt.month

future_fe["quarter"] = future_fe["date"].dt.quarter

future_fe["week_of_year"] = (
    future_fe["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

future_fe["day_of_week"] = future_fe["date"].dt.dayofweek

future_fe["day_of_month"] = future_fe["date"].dt.day

future_fe["is_weekend"] = (
        future_fe["day_of_week"] >= 5
).astype(int)

# ==========================================================
# Validate Calendar Features
# ==========================================================

print("=" * 60)
print("CALENDAR FEATURES")
print("=" * 60)

print("\nTraining Data")
display(
    sales_fe[
        [
            "date",
            *calendar_features
        ]
    ].head()
)

print("\nFuture Data")
display(
    future_fe[
        [
            "date",
            *calendar_features
        ]
    ].head()
)

# ==========================================================
# Check Missing Values
# ==========================================================

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

print("\nTraining")
print(
    sales_fe[calendar_features]
    .isna()
    .sum()
)

print("\nFuture")
print(
    future_fe[calendar_features]
    .isna()
    .sum()
)

CALENDAR FEATURES

Training Data


,date,year,month,quarter,week_of_year,day_of_week,day_of_month,is_weekend,is_month_start,is_month_end,is_quarter_end
0,2015-07-19,2015,7,3,29,6,19,1,0,0,0
1,2015-07-19,2015,7,3,29,6,19,1,0,0,0
2,2015-07-19,2015,7,3,29,6,19,1,0,0,0
3,2015-07-19,2015,7,3,29,6,19,1,0,0,0
4,2015-07-19,2015,7,3,29,6,19,1,0,0,0



Future Data


,date,year,month,quarter,week_of_year,day_of_week,day_of_month,is_weekend,is_month_start,is_month_end,is_quarter_end
0,2015-09-17,2015,9,3,38,3,17,0,0,0,0
1,2015-09-17,2015,9,3,38,3,17,0,0,0,0
2,2015-09-17,2015,9,3,38,3,17,0,0,0,0
3,2015-09-17,2015,9,3,38,3,17,0,0,0,0
4,2015-09-17,2015,9,3,38,3,17,0,0,0,0


MISSING VALUES

Training
year              0
month             0
quarter           0
week_of_year      0
day_of_week       0
day_of_month      0
is_weekend        0
is_month_start    0
is_month_end      0
is_quarter_end    0
dtype: int64

Future
year              0
month             0
quarter           0
week_of_year      0
day_of_week       0
day_of_month      0
is_weekend        0
is_month_start    0
is_month_end      0
is_quarter_end    0
dtype: int64


In [5]:
# ==========================================================
#  Lag Features
# ==========================================================
# Lag features provide historical sales information so the
# model can learn temporal dependencies.
#
# Features:
# lag_1   : Sales from previous day
# lag_7   : Sales from same weekday last week
# lag_14  : Sales from two weeks ago
# lag_21  : Sales from three weeks ago
# lag_28  : Sales from four weeks ago
#
# Lag features are among the most important predictors for
# tree-based forecasting models.
# ==========================================================

# ==========================================================
# Generate Lag Features
# ==========================================================

# Important: sort by store and date first so each store's
# lag is computed from its own previous day, not from the
# previous row in the raw file order.
sales_fe = sales_fe.sort_values(["store_id", "date"]).reset_index(drop=True)

lag_features = [1, 7, 14, 21, 28]

for lag in lag_features:
    sales_fe[f"lag_{lag}"] = (
        sales_fe.groupby("store_id", sort=False)["sales"]
        .shift(lag)
    )

promo_lags = [1,7]

for lag in promo_lags:

    sales_fe[f"promo_lag_{lag}"] = (
        sales_fe
        .groupby("store_id")["promo"]
        .shift(lag)
    )


# ==========================================================
# Validate Lag Features
# ==========================================================

lag_columns = [f"lag_{lag}" for lag in lag_features]

display(
    sales_fe[
        [
            "store_id",
            "date",
            "sales",
            *lag_columns
        ]
    ].head(35)
)

# ==========================================================
# Check Missing Values
# ==========================================================

print(sales_fe[lag_columns].isna().sum())
print("\nNon-null lag_1 values:", sales_fe["lag_1"].notna().sum())

,store_id,date,sales,lag_1,lag_7,lag_14,lag_21,lag_28
0,store_1,2013-01-07,7176,NaN,NaN,NaN,NaN,NaN
1,store_1,2013-01-08,5580,7176.0,NaN,NaN,NaN,NaN
2,store_1,2013-01-09,5471,5580.0,NaN,NaN,NaN,NaN
3,store_1,2013-01-10,4892,5471.0,NaN,NaN,NaN,NaN
4,store_1,2013-01-11,4881,4892.0,NaN,NaN,NaN,NaN
5,store_1,2013-01-12,4952,4881.0,NaN,NaN,NaN,NaN
6,store_1,2013-01-13,0,4952.0,NaN,NaN,NaN,NaN
7,store_1,2013-01-14,4717,0.0,7176.0,NaN,NaN,NaN
8,store_1,2013-01-15,3900,4717.0,5580.0,NaN,NaN,NaN
9,store_1,2013-01-16,4008,3900.0,5471.0,NaN,NaN,NaN


lag_1       676
lag_7      4732
lag_14     9464
lag_21    14196
lag_28    18928
dtype: int64

Non-null lag_1 values: 623948


In [6]:
# ==========================================================
# Rolling Features: Mean and Std
# ==========================================================
# Lag features provide a single historical observation,
# while rolling features summarize recent sales behaviour.
#
# Features:
# rolling_mean_7  : Average sales over previous 7 days
# rolling_mean_14 : Average sales over previous 14 days
# rolling_mean_28 : Average sales over previous 28 days
#
# Rolling features help machine learning models capture
# short-term trends and smooth out daily fluctuations.
# ==========================================================

rolling_windows = [7, 14, 21, 28]

for window in rolling_windows:

    sales_fe[f"rolling_mean_{window}"] = (
        sales_fe
        .groupby("store_id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    sales_fe[f"rolling_std_{window}"] = (
        sales_fe
        .groupby("store_id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

#Due to Data leakage, we shift the rolling mean by 1 day to ensure that 
#the model does not have access to future information when making predictions.

# ==========================================================
# Validate Rolling Features
# ==========================================================

rolling_columns = []

for window in rolling_windows:

    rolling_columns.extend([
        f"rolling_mean_{window}",
        f"rolling_std_{window}"
    ])

    sales_fe[f"rolling_median_{window}"] = (
        sales_fe
        .groupby("store_id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).median()
        )
    )

    rolling_columns.append(
        f"rolling_median_{window}"
    )

for window in rolling_windows:

    sales_fe[f"rolling_max_{window}"] = (
        sales_fe
        .groupby("store_id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).max()
        )
    )

    sales_fe[f"rolling_min_{window}"] = (
        sales_fe
        .groupby("store_id")["sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).min()
        )
    )

for window in [7,14]:

    sales_fe[f"promo_mean_{window}"] = (
        sales_fe
        .groupby("store_id")["promo"]
        .transform(
            lambda x:
            x.shift(1)
            .rolling(window)
            .mean()
        )
    )

store = "store_1"

display(
    sales_fe.loc[
        sales_fe["store_id"] == store,
        [
            "date",
            "sales",
            *rolling_columns
        ]
    ].head(35)
)

# ==========================================================
# Check Missing Values
# ==========================================================

print(sales_fe[rolling_columns].isna().sum())



,date,sales,rolling_mean_7,rolling_std_7,rolling_median_7,rolling_mean_14,rolling_std_14,rolling_median_14,rolling_mean_21,rolling_std_21,rolling_median_21,rolling_mean_28,rolling_std_28,rolling_median_28
0,2013-01-07,7176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-08,5580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-01-09,5471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-01-10,4892,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-01-11,4881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2013-01-12,4952,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2013-01-13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2013-01-14,4717,4707.428571,2225.689396,4952.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2013-01-15,3900,4356.142857,1947.844743,4892.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2013-01-16,4008,4116.142857,1874.016847,4881.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


rolling_mean_7        4732
rolling_std_7         4732
rolling_median_7      4732
rolling_mean_14       9464
rolling_std_14        9464
rolling_median_14     9464
rolling_mean_21      14196
rolling_std_21       14196
rolling_median_21    14196
rolling_mean_28      18928
rolling_std_28       18928
rolling_median_28    18928
dtype: int64


In [7]:
# ==========================================================
# Generate Historical Features for Future Dataset
# ==========================================================
# Use the latest historical information of each store to
# create lag and rolling features for future prediction.
# Future sales are unknown, therefore no recursive forecasting
# is used.
# ==========================================================

lag_features = [1, 7, 14, 21, 28]
rolling_windows = [7, 14, 21, 28]

# Sort historical data
sales_fe = (
    sales_fe
    .sort_values(["store_id", "date"])
    .reset_index(drop=True)
)

for store in future_fe["store_id"].unique():

    history = (
        sales_fe.loc[
            sales_fe["store_id"] == store
            ]
        .sort_values("date")
    )

    # ---------- Lag Features ----------
    for lag in lag_features:
        future_fe.loc[
            future_fe["store_id"] == store,
            f"lag_{lag}"
        ] = history["sales"].iloc[-lag]

    # ---------- Promo Lag ----------
    for lag in [1, 7]:
        future_fe.loc[
            future_fe["store_id"] == store,
            f"promo_lag_{lag}"
        ] = history["promo"].iloc[-lag]

    # ---------- Rolling Mean ----------
    for window in rolling_windows:
        future_fe.loc[
            future_fe["store_id"] == store,
            f"rolling_mean_{window}"
        ] = history["sales"].tail(window).mean()

        future_fe.loc[
            future_fe["store_id"] == store,
            f"rolling_std_{window}"
        ] = history["sales"].tail(window).std()

        future_fe.loc[
            future_fe["store_id"] == store,
            f"rolling_median_{window}"
        ] = history["sales"].tail(window).median()

        future_fe.loc[
            future_fe["store_id"] == store,
            f"rolling_max_{window}"
        ] = history["sales"].tail(window).max()

        future_fe.loc[
            future_fe["store_id"] == store,
            f"rolling_min_{window}"
        ] = history["sales"].tail(window).min()

    # ---------- Promo Rolling ----------
    for window in [7, 14]:
        future_fe.loc[
            future_fe["store_id"] == store,
            f"promo_mean_{window}"
        ] = history["promo"].tail(window).mean()

In [8]:
print(future_fe[
    [
        "lag_1",
        "lag_7",
        "rolling_mean_7",
        "promo_lag_1"
    ]
].head())

   lag_1    lag_7  rolling_mean_7  promo_lag_1
0    0.0   5054.0     4078.285714          0.0
1    0.0   9462.0     6513.714286          0.0
2    0.0  13250.0     9087.571429          0.0
3    0.0   9117.0     6463.285714          0.0
4    0.0  10399.0     6599.714286          0.0


In [9]:
print(
    future_fe[
        [
            "lag_1",
            "lag_7",
            "rolling_mean_7",
            "promo_lag_1",
            "promo_mean_7"
        ]
    ].describe()
)

              lag_1         lag_7  rolling_mean_7  promo_lag_1  promo_mean_7
count  40560.000000  40560.000000    40560.000000      40560.0  4.056000e+04
mean     349.173077   9614.036982     6666.911665          0.0  7.142857e-01
std     2188.380659   3133.696144     2286.787000          0.0  2.220473e-16
min        0.000000   3658.000000     2697.857143          0.0  7.142857e-01
25%        0.000000   7620.750000     5160.928571          0.0  7.142857e-01
50%        0.000000   9133.500000     6305.714286          0.0  7.142857e-01
75%        0.000000  11287.750000     7675.964286          0.0  7.142857e-01
max    31665.000000  28156.000000    22853.857143          0.0  7.142857e-01


In [10]:
future_fe[
    [
        "lag_1",
        "lag_7",
        "rolling_mean_7",
        "promo_lag_1",
        "promo_mean_7"
    ]
].isna().sum()

lag_1             0
lag_7             0
rolling_mean_7    0
promo_lag_1       0
promo_mean_7      0
dtype: int64

In [11]:
#interaction feature
sales_fe["promo_weekend"] = (
    sales_fe["promo"] *
    sales_fe["is_weekend"]
)

future_fe["promo_weekend"] = (
    future_fe["promo"] *
    future_fe["is_weekend"]
)

sales_fe["promo_schoolholiday"] = (
    sales_fe["promo"] *
    sales_fe["school_holiday"]
)

future_fe["promo_schoolholiday"] = (
    future_fe["promo"] *
    future_fe["school_holiday"]
)

sales_fe["promo_stateholiday"] = (
    sales_fe["promo"] *
    (sales_fe["state_holiday"] != "0").astype(int)
)

future_fe["promo_stateholiday"] = (
    future_fe["promo"] *
    (future_fe["state_holiday"] != "0").astype(int)
)

In [12]:
# ==========================================================
# One-Hot Encoding
# ==========================================================
# Tree-based machine learning models require numerical inputs.
# Convert categorical variables into binary indicator variables
# using one-hot encoding.
#
# Features:
# store_type
# assortment
# state_holiday
# ==========================================================

categorical_features = [
    "store_type",
    "assortment",
    "state_holiday"
]

sales_fe = pd.get_dummies(
    sales_fe,
    columns=categorical_features,
    dtype=int
)

future_fe = pd.get_dummies(
    future_fe,
    columns=categorical_features,
    dtype=int
)

# ==========================================================
# Align Training and Future Features
# ==========================================================
# Ensure both datasets contain exactly the same feature columns.
# Missing columns in the future dataset are filled with zeros.

sales_fe, future_fe = sales_fe.align(
    future_fe,
    join="left",
    axis=1,
    fill_value=0
)

# Restore the target variable after alignment.
sales_fe["sales"] = sales_clean["sales"]

# ==========================================================
# Validate One-Hot Encoding
# ==========================================================

encoded_columns = [
    col for col in sales_fe.columns
    if col.startswith("store_type_")
       or col.startswith("assortment_")
       or col.startswith("state_holiday_")
]

print("=" * 60)
print("ONE-HOT ENCODED FEATURES")
print("=" * 60)

print(encoded_columns)

print("\nTraining sample")
display(sales_fe[encoded_columns].head())

print("\nFuture sample")
display(future_fe[encoded_columns].head())

print("\nMissing values (Training)")
print(sales_fe[encoded_columns].isna().sum())

print("\nMissing values (Future)")
print(future_fe[encoded_columns].isna().sum())

ONE-HOT ENCODED FEATURES
['store_type_a', 'store_type_b', 'store_type_c', 'store_type_d', 'assortment_a', 'assortment_b', 'assortment_c', 'state_holiday_0', 'state_holiday_a', 'state_holiday_b', 'state_holiday_c']

Training sample


,store_type_a,store_type_b,store_type_c,store_type_d,assortment_a,assortment_b,assortment_c,state_holiday_0,state_holiday_a,state_holiday_b,state_holiday_c
0,0,0,1,0,1,0,0,1,0,0,0
1,0,0,1,0,1,0,0,1,0,0,0
2,0,0,1,0,1,0,0,1,0,0,0
3,0,0,1,0,1,0,0,1,0,0,0
4,0,0,1,0,1,0,0,1,0,0,0



Future sample


,store_type_a,store_type_b,store_type_c,store_type_d,assortment_a,assortment_b,assortment_c,state_holiday_0,state_holiday_a,state_holiday_b,state_holiday_c
0,0,0,1,0,1,0,0,1,0,0,0
1,1,0,0,0,1,0,0,1,0,0,0
2,1,0,0,0,0,0,1,1,0,0,0
3,1,0,0,0,1,0,0,1,0,0,0
4,1,0,0,0,0,0,1,1,0,0,0



Missing values (Training)
store_type_a       0
store_type_b       0
store_type_c       0
store_type_d       0
assortment_a       0
assortment_b       0
assortment_c       0
state_holiday_0    0
state_holiday_a    0
state_holiday_b    0
state_holiday_c    0
dtype: int64

Missing values (Future)
store_type_a       0
store_type_b       0
store_type_c       0
store_type_d       0
assortment_a       0
assortment_b       0
assortment_c       0
state_holiday_0    0
state_holiday_a    0
state_holiday_b    0
state_holiday_c    0
dtype: int64


In [13]:
# ==========================================================
# Final Feature Validation
# ==========================================================
# Perform a final check before modelling.

print("=" * 60)
print("FINAL FEATURE VALIDATION")
print("=" * 60)

print("\nTraining shape:")
print(sales_fe.shape)

print("\nFuture shape:")
print(future_fe.shape)

print("\nTraining missing values:")
print(sales_fe.isna().sum().sort_values(ascending=False).head(20))

print("\nFuture missing values:")
print(future_fe.isna().sum().sort_values(ascending=False).head(20))

print("\nTraining columns:")
print(sales_fe.columns.tolist())

print("\nFuture columns:")
print(future_fe.columns.tolist())

FINAL FEATURE VALIDATION

Training shape:
(624624, 61)

Future shape:
(40560, 61)

Training missing values:
rolling_min_28       18928
rolling_median_28    18928
rolling_std_28       18928
rolling_mean_28      18928
rolling_max_28       18928
lag_28               18928
rolling_std_21       14196
rolling_median_21    14196
rolling_mean_21      14196
rolling_min_21       14196
lag_21               14196
rolling_max_21       14196
rolling_min_14        9464
rolling_max_14        9464
promo_mean_14         9464
rolling_median_14     9464
lag_14                9464
rolling_std_14        9464
rolling_mean_14       9464
promo_lag_7           4732
dtype: int64

Future missing values:
customers              40560
store_id                   0
promo_mean_14              0
rolling_median_7           0
rolling_median_14          0
rolling_median_21          0
rolling_median_28          0
rolling_max_7              0
rolling_min_7              0
rolling_max_14             0
rolling_min_14           

In [14]:
sales_fe["store_id"] = (
    sales_fe["store_id"]
    .str.replace("store_", "")
    .astype(int)
)

future_fe["store_id"] = (
    future_fe["store_id"]
    .str.replace("store_", "")
    .astype(int)
)

In [15]:
sales_fe.to_csv(
    processed_dir / "sales_fe.csv",
    index=False
)

future_fe.to_csv(
    processed_dir / "future_fe.csv",
    index=False
)

In [17]:
future_fe[
    [
        "lag_1",
        "lag_7",
        "rolling_mean_7",
        "promo_lag_1",
        "promo_mean_7"
    ]
].describe()

,lag_1,lag_7,rolling_mean_7,promo_lag_1,promo_mean_7
count,40560.000000,40560.000000,40560.000000,40560.0,4.056000e+04
mean,349.173077,9614.036982,6666.911665,0.0,7.142857e-01
std,2188.380659,3133.696144,2286.787000,0.0,2.220473e-16
min,0.000000,3658.000000,2697.857143,0.0,7.142857e-01
25%,0.000000,7620.750000,5160.928571,0.0,7.142857e-01
50%,0.000000,9133.500000,6305.714286,0.0,7.142857e-01
75%,0.000000,11287.750000,7675.964286,0.0,7.142857e-01
max,31665.000000,28156.000000,22853.857143,0.0,7.142857e-01


In [18]:
print(sales_fe.groupby("store_id")["sales"].last().describe())

print(sales_fe.groupby("store_id")["promo"].last().value_counts())

count      676.000000
mean      5966.390533
std       3924.288846
min          0.000000
25%       4014.500000
50%       6033.500000
75%       8075.000000
max      34814.000000
Name: sales, dtype: float64
promo
0    676
Name: count, dtype: int64


In [19]:
print(sales_fe["store_id"].nunique())

676


In [20]:
sales_fe.groupby("store_id")[["sales", "open"]].last().head(20)

,sales,open
store_id,,
1,7669,0
2,6529,0
3,6559,0
4,7111,0
5,10583,0
6,5134,0
7,5328,0
8,21987,0
9,0,0
